# NWP P2P Federated — Training Notebook (v11)

Trains the `word_lstm_v1` LSTM language model used by the main app.
Outputs a `lstm_weights.json` file that can be loaded by the app via
`LstmWordModel.load_external_weights(path)`.

**Colab-compatible.** Uses only PyTorch + standard library.

---
## Pipeline
1. Upload / paste your dataset
2. Preprocess and build vocabulary
3. Train LSTM (optionally with n-gram baseline)
4. Evaluate
5. Export weights → `lstm_weights.json`

In [ ]:
# ── 1. Install dependencies (Colab) ──────────────────────────────────────────
# PyTorch is pre-installed on Colab. Nothing extra needed.
import sys
print(f'Python {sys.version}')
import torch
print(f'PyTorch {torch.__version__} · CUDA: {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {DEVICE}')

In [ ]:
# ── 2. Dataset upload ─────────────────────────────────────────────────────────
# Option A: paste text directly
INLINE_TEXT = """
The quick brown fox jumps over the lazy dog.
Federated learning enables privacy-preserving model training.
Next word prediction improves typing speed significantly.
"""

# Option B: upload a .txt file in Colab
USE_UPLOAD = False  # Set True to upload your own file
if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    raw_text = uploaded[filename].decode('utf-8')
else:
    raw_text = INLINE_TEXT

print(f'Dataset: {len(raw_text):,} chars')

In [ ]:
# ── 3. Preprocessing & vocabulary ────────────────────────────────────────────
import re
from collections import Counter

_WORD_RE = re.compile(r"[a-zA-Z']+|[0-9]+")

def tokenize(text: str) -> list:
    return [t.lower() for t in _WORD_RE.findall(text) if t]

# ── Hyperparameters ───────────────────────────────────────────────────────────
MAX_VOCAB   = 2500   # must match app value (hybrid_model.py _build_vocab)
MAX_SEQ_LEN = 16     # must match LstmWordModel.MAX_SEQ_LEN
EMBED_DIM   = 96     # must match _WordLSTM embed_dim
HIDDEN_DIM  = 160    # must match _WordLSTM hidden_dim
BATCH_SIZE  = 32
EPOCHS      = 10
LR          = 3e-3
GRAD_CLIP   = 1.0    # must match training_agent clip value

# Build vocabulary — include the same special tokens as the app
all_words = tokenize(raw_text)
freq = Counter(all_words)

# Fixed special tokens (positions must match app's _build_vocab output)
word_to_id = {'<pad>': 0, '<unk>': 1, '<bos>': 2}

# Add top-N words by frequency (excluding specials)
for word, _ in freq.most_common(MAX_VOCAB - len(word_to_id)):
    if word not in word_to_id:
        word_to_id[word] = len(word_to_id)

id_to_word = [''] * len(word_to_id)
for w, i in word_to_id.items():
    id_to_word[i] = w

VOCAB_SIZE = len(word_to_id)
UNK_ID = word_to_id['<unk>']
BOS_ID = word_to_id['<bos>']

print(f'Vocabulary size: {VOCAB_SIZE}')
print(f'Total tokens: {len(all_words):,}')
print(f'Top 10 words: {[w for w,_ in freq.most_common(10)]}')

In [ ]:
# ── 4. Build training sequences ───────────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader

def encode(words):
    return [word_to_id.get(w, UNK_ID) for w in words]

# Split raw text into sentences (simple punctuation split)
sentences = [s.strip() for s in re.split(r'[.!?\n]+', raw_text) if s.strip()]

class NWPDataset(Dataset):
    def __init__(self, sentences, max_len=MAX_SEQ_LEN):
        self.pairs = []
        for sent in sentences:
            words = tokenize(sent)
            if len(words) < 2:
                continue
            ids = [BOS_ID] + encode(words)
            ids = ids[-(max_len + 1):]  # truncate to max_len+1
            self.pairs.append(ids)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        ids = self.pairs[idx]
        return torch.tensor(ids[:-1], dtype=torch.long), torch.tensor(ids[1:], dtype=torch.long)

def collate_fn(batch):
    xs, ys = zip(*batch)
    max_len = max(x.size(0) for x in xs)
    xs_padded = torch.zeros(len(xs), max_len, dtype=torch.long)
    ys_padded = torch.full((len(ys), max_len), -100, dtype=torch.long)  # -100 = ignore_index
    for i, (x, y) in enumerate(zip(xs, ys)):
        xs_padded[i, :x.size(0)] = x
        ys_padded[i, :y.size(0)] = y
    return xs_padded, ys_padded

dataset = NWPDataset(sentences)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
print(f'Training samples: {len(dataset)}')

In [ ]:
# ── 5. Model definition (mirrors app's _WordLSTM exactly) ────────────────────
import torch.nn as nn

class WordLSTM(nn.Module):
    """Must match _WordLSTM in hybrid_model.py exactly for weight compatibility."""
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm  = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.head  = nn.Linear(hidden_dim, vocab_size)

    def forward(self, tokens):
        x = self.embed(tokens)
        out, _ = self.lstm(x)
        return self.head(out)

model = WordLSTM(VOCAB_SIZE).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
loss_fn   = nn.CrossEntropyLoss(ignore_index=-100)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,} (~{total_params*4/1e6:.1f} MB)')

In [ ]:
# ── 6. Training loop ──────────────────────────────────────────────────────────
import time

history = []
model.train()

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    total_loss, n_batches = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)                        # (B, T, V)
        # Flatten for CE: (B*T, V) vs (B*T,)
        loss = loss_fn(logits.view(-1, VOCAB_SIZE), y.view(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        # Gradient clipping — same value as app training path
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item()
        n_batches  += 1
    avg_loss = total_loss / max(n_batches, 1)
    elapsed  = time.time() - t0
    history.append({'epoch': epoch, 'loss': avg_loss})
    print(f'Epoch {epoch:2d}/{EPOCHS} | loss {avg_loss:.4f} | {elapsed:.1f}s')

model.eval()
print('\nTraining complete.')

In [ ]:
# ── 7. (Optional) N-gram baseline for comparison ─────────────────────────────
from collections import defaultdict

bigrams  = defaultdict(Counter)
trigrams = defaultdict(Counter)

for sent in sentences:
    words = tokenize(sent)
    for i, w in enumerate(words):
        if i >= 1: bigrams[words[i-1]][w] += 1
        if i >= 2: trigrams[(words[i-2], words[i-1])][w] += 1

def ngram_predict(context, k=5):
    ctx = [w.lower() for w in context]
    # Try trigram first
    if len(ctx) >= 2:
        candidates = trigrams.get(tuple(ctx[-2:]), {})
        if candidates:
            return candidates.most_common(k)
    # Bigram fallback
    candidates = bigrams.get(ctx[-1] if ctx else '', {})
    return candidates.most_common(k)

# Quick sanity check
test_ctx = ['the']
print(f'N-gram predictions after {test_ctx}: {ngram_predict(test_ctx)}')

In [ ]:
# ── 8. Quick inference test ───────────────────────────────────────────────────
def lstm_predict(context_words, k=5):
    ids = [word_to_id.get(w.lower(), UNK_ID) for w in context_words[-MAX_SEQ_LEN:]] or [BOS_ID]
    x = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logits = model(x)[0, -1].float()
        probs  = torch.softmax(logits, dim=-1)
        top    = torch.topk(probs, k=min(k+8, VOCAB_SIZE))
    results = []
    for idx, score in zip(top.indices.tolist(), top.values.tolist()):
        word = id_to_word[idx]
        if word.startswith('<'): continue
        results.append((word, round(score, 4)))
        if len(results) >= k: break
    return results

test_inputs = [['the'], ['federated', 'learning'], ['please', 'let', 'me']]
for ctx in test_inputs:
    preds = lstm_predict(ctx)
    print(f'After {ctx}: {preds}')

In [ ]:
# ── 9. Export weights (app-compatible format) ────────────────────────────────
# Output: lstm_weights.json
# Loaded by app via: LstmWordModel.load_external_weights('lstm_weights.json')

import base64, gzip, io, json

buf = io.BytesIO()
torch.save(model.state_dict(), buf)
raw = gzip.compress(buf.getvalue())

payload = {
    'format': 'torch_state_gzip_b64',
    'arch':   'word_lstm_v1',          # must match LstmWordModel.apply_peer_state check
    'train_steps': EPOCHS * len(loader),
    'vocab_size': VOCAB_SIZE,
    'embed_dim':  EMBED_DIM,
    'hidden_dim': HIDDEN_DIM,
    'training_loss_history': history,
    'blob': base64.b64encode(raw).decode('ascii'),
}

output_path = 'lstm_weights.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(payload, f)

import os
size_mb = os.path.getsize(output_path) / 1e6
print(f'Exported: {output_path} ({size_mb:.2f} MB)')
print('Load in app: slot.model_agent.lstm.load_external_weights("lstm_weights.json")')

In [ ]:
# ── 10. Download from Colab ───────────────────────────────────────────────────
try:
    from google.colab import files
    files.download('lstm_weights.json')
    print('Download started.')
except ImportError:
    print('Not in Colab — file saved locally as lstm_weights.json')

## Loading weights in the app

```python
# In Python — after starting the server:
from nwp_app.trainer.model_registry import ModelRegistry
registry = ModelRegistry(data_dir='data/trainer')
slot = registry.get('default')
ok = slot.model_agent.lstm.load_external_weights('lstm_weights.json')
print('Loaded:', ok)
```

Or via the `/local/ngram/save` API endpoint after manually placing the file in
`data/trainer/lstm_default.json`.

## Framework notes
- **PyTorch** — chosen for its lightweight CPU inference, dynamic graphs, and
  easy state_dict serialisation (ideal for federated delta sharing).
- Model size: ~1–3 MB (compressed). Well within the <10 MB on-device target.
- For larger datasets: increase `MAX_VOCAB`, `HIDDEN_DIM`, and `EPOCHS`.
- For faster training on Colab: enable GPU (Runtime → Change runtime type → T4 GPU).

---
## 11. Model Evaluation — Metrics

We measure the trained LSTM using five metrics standard for next-word prediction:

| Metric | What it tells you |
| --- | --- |
| **Perplexity** | How surprised the model is on held-out text. Lower = better. |
| **Top-1 Accuracy** | Fraction of steps where the top prediction is correct. |
| **Top-5 Accuracy** | Fraction of steps where the correct word is in the top-5 predictions. |
| **MRR** | Mean Reciprocal Rank — rewards placing the correct word higher in the ranked list. |
| **BLEU-1** | Unigram overlap between generated completions and references. |

In [ ]:
# ── 11a. Build a held-out evaluation split ────────────────────────────────────────────
import random, math

random.seed(42)
all_sentences = [s.strip() for s in re.split(r'[.!?\n]+', raw_text) if s.strip()]

# 80/20 train/eval split
random.shuffle(all_sentences)
split = max(1, int(0.8 * len(all_sentences)))
eval_sentences = all_sentences[split:] if len(all_sentences) > 1 else all_sentences

# Build (context_ids, target_id) pairs
eval_pairs = []
for sent in eval_sentences:
    words = tokenize(sent)
    if len(words) < 2:
        continue
    ids = [BOS_ID] + [word_to_id.get(w, UNK_ID) for w in words]
    for t in range(1, len(ids)):
        ctx = ids[max(0, t - MAX_SEQ_LEN):t]
        eval_pairs.append((ctx, ids[t]))

print(f'Eval sentences       : {len(eval_sentences)}')
print(f'Eval (ctx, tgt) pairs: {len(eval_pairs)}')

In [ ]:
# ── 11b. Perplexity · Top-1 · Top-5 · MRR ────────────────────────────────────────────
import torch

model.eval()
BATCH_EVAL = 64
K_MRR = 10   # rank within top-K for MRR

total_nll, total_steps = 0.0, 0
top1_correct = top5_correct = 0
mrr_sum = 0.0

def pad_batch(ctx_list):
    """Left-pad variable-length context lists to the same length."""
    max_l = max(len(c) for c in ctx_list)
    padded = torch.zeros(len(ctx_list), max_l, dtype=torch.long)
    for i, c in enumerate(ctx_list):
        padded[i, max_l - len(c):] = torch.tensor(c, dtype=torch.long)
    return padded

with torch.no_grad():
    for start in range(0, len(eval_pairs), BATCH_EVAL):
        batch = eval_pairs[start : start + BATCH_EVAL]
        ctxs  = [p[0] for p in batch]
        tgts  = torch.tensor([p[1] for p in batch], dtype=torch.long).to(DEVICE)

        x      = pad_batch(ctxs).to(DEVICE)
        logits = model(x)[:, -1, :].float()          # (B, V)
        log_p  = torch.log_softmax(logits, dim=-1)

        # Perplexity
        total_nll   += (-log_p[torch.arange(len(batch)), tgts]).sum().item()
        total_steps += len(batch)

        # Top-1 / Top-5
        top5 = torch.topk(logits, k=min(5, VOCAB_SIZE), dim=-1).indices
        top1_correct += (top5[:, 0] == tgts).sum().item()
        top5_correct += (top5 == tgts.unsqueeze(1)).any(1).sum().item()

        # MRR
        topk = torch.topk(logits, k=min(K_MRR, VOCAB_SIZE), dim=-1).indices
        for i, tgt in enumerate(tgts.tolist()):
            hits = (topk[i] == tgt).nonzero(as_tuple=False)
            if hits.numel():
                mrr_sum += 1.0 / (hits[0, 0].item() + 1)

perplexity = math.exp(total_nll / max(total_steps, 1))
top1_acc   = top1_correct / max(total_steps, 1)
top5_acc   = top5_correct / max(total_steps, 1)
mrr        = mrr_sum      / max(total_steps, 1)

print('=' * 46)
print(f'  Perplexity            : {perplexity:>10.2f}')
print(f'  Top-1 Accuracy        : {top1_acc:>10.4f}  ({top1_acc*100:.1f}%)')
print(f'  Top-5 Accuracy        : {top5_acc:>10.4f}  ({top5_acc*100:.1f}%)')
print(f'  MRR (top-{K_MRR})         : {mrr:>10.4f}')
print('=' * 46)
print(f'  Eval steps            : {total_steps}')

In [ ]:
# ── 11c. BLEU-1 (greedy sentence completion) ───────────────────────────────────────
from collections import Counter as _Counter

def bleu1_sentence(hypothesis: list, reference: list) -> float:
    """Sentence-level BLEU-1 with brevity penalty."""
    if not hypothesis:
        return 0.0
    ref_cnt   = _Counter(reference)
    clip      = sum(min(c, ref_cnt[w]) for w, c in _Counter(hypothesis).items())
    precision = clip / len(hypothesis)
    bp        = math.exp(1 - len(reference) / len(hypothesis)) if len(hypothesis) > len(reference) else 1.0
    return bp * precision

bleu1_scores = []
model.eval()
with torch.no_grad():
    for sent in eval_sentences:
        words = tokenize(sent)
        if len(words) < 2:
            continue
        ids = [BOS_ID] + [word_to_id.get(w, UNK_ID) for w in words]
        hypothesis = []
        for t in range(1, len(ids)):
            ctx = ids[max(0, t - MAX_SEQ_LEN):t]
            x   = torch.tensor([ctx], dtype=torch.long).to(DEVICE)
            pred_id = model(x)[0, -1].float().argmax().item()
            hypothesis.append(id_to_word[pred_id] if pred_id < len(id_to_word) else '<unk>')
        bleu1_scores.append(bleu1_sentence(hypothesis, words))

avg_bleu1 = sum(bleu1_scores) / max(len(bleu1_scores), 1)
print(f'  BLEU-1 (greedy completion): {avg_bleu1:.4f}  ({avg_bleu1*100:.1f}%)')

In [ ]:
# ── 11d. Summary table + plots ──────────────────────────────────────────────────
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

print('\n── Final Metric Summary ───────────────────────────────────────')
metrics_summary = [
    ('Perplexity          (↓)', f'{perplexity:.2f}'),
    ('Top-1 Accuracy      (↑)', f'{top1_acc*100:.1f}%'),
    ('Top-5 Accuracy      (↑)', f'{top5_acc*100:.1f}%'),
    (f'MRR (top-{K_MRR})       (↑)', f'{mrr:.4f}'),
    ('BLEU-1              (↑)', f'{avg_bleu1:.4f}'),
]
for name, val in metrics_summary:
    print(f'  {name:32s}: {val}')
print('─' * 58)

if HAS_MPL and history:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Training loss curve
    epochs_x = [h['epoch'] for h in history]
    losses   = [h['loss']  for h in history]
    axes[0].plot(epochs_x, losses, marker='o', linewidth=2, color='steelblue')
    axes[0].set_title('Training Loss (Cross-Entropy)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

    # Metric bar chart (0-1 scale metrics only)
    bar_labels = ['Top-1 Acc', 'Top-5 Acc', f'MRR@{K_MRR}', 'BLEU-1']
    bar_vals   = [top1_acc, top5_acc, mrr, avg_bleu1]
    colors     = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
    bars = axes[1].bar(bar_labels, bar_vals, color=colors, alpha=0.85, edgecolor='white')
    axes[1].set_ylim(0, 1.05)
    axes[1].set_title('Evaluation Metrics (higher = better)')
    axes[1].set_ylabel('Score')
    axes[1].grid(True, axis='y', alpha=0.3)
    for bar, val in zip(bars, bar_vals):
        axes[1].text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    plt.suptitle(f'NWP LSTM — Perplexity: {perplexity:.1f}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('eval_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved: eval_metrics.png')

### Interpreting the metrics

| Metric | Typical range (small dataset) | Notes |
| --- | --- | --- |
| **Perplexity** | 10 – 200 | Near vocab size = near-random; near 1 = overfit |
| **Top-1 Acc** | 5 – 40 % | Hard — natural language has high branching factor |
| **Top-5 Acc** | 20 – 70 % | Most relevant for autocomplete UX |
| **MRR** | 0.10 – 0.50 | Penalises correct words ranked low |
| **BLEU-1** | 0.10 – 0.50 | Rises with dataset size and training epochs |

> **Tip:** Low top-5 + high perplexity → need more data or longer training.  
> Low BLEU-1 despite decent top-1 → model is inconsistent across full sentences.